# Diagnóstico inicial de calidad de datos — FiveM Analytics

**Proyecto:** Análisis de rendimiento, retención de jugadores y economía virtual en servidores de roleplay de FiveM GTA V.

**Pregunta analítica:** ¿Cuáles son las actividades, trabajos y zonas del mapa con mayor concurrencia de jugadores dentro del servidor de FiveM, y cómo influye el balance económico (ingresos vs. gastos virtuales) en la retención y tiempo promedio de juego diario de la comunidad?

Por defecto, inspecciona cuatro CSV generados para simular un servidor FiveM y un dataset público de Kaggle. El archivo de Kaggle fue creado sintéticamente por su autor; ninguno de estos archivos es telemetría observada de un servidor real. Cambia `DATA_MODE` a `"real"` solo si agregas exportaciones auténticas autorizadas y anonimizadas en `data/raw/`. Este Notebook diagnostica la calidad, no limpia ni corrige automáticamente.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_MODE = "synthetic"  # Usa "real" cuando tengas ambas fuentes auténticas.

if DATA_MODE == "synthetic":
    SOURCE_DIRS = {
        "FiveM sintético (demostración)": PROJECT_ROOT / "data" / "examples" / "fivem",
        "Kaggle (dataset público sintético)": PROJECT_ROOT / "data" / "examples" / "kaggle",
    }
    print("AVISO: FiveM es un escenario sintético; el dataset público de Kaggle también se declara sintético por su autor.")
elif DATA_MODE == "real":
    SOURCE_DIRS = {
        "FiveM / MySQL / txAdmin": PROJECT_ROOT / "data" / "raw" / "fivem",
        "Kaggle / telemetría y retención": PROJECT_ROOT / "data" / "raw" / "kaggle",
    }
else:
    raise ValueError("DATA_MODE debe ser 'synthetic' o 'real'.")

SUPPORTED_SUFFIXES = {".csv"}

print(f"Raíz del proyecto: {PROJECT_ROOT}")
for source, folder in SOURCE_DIRS.items():
    print(f"{source}: {folder}")

: 

In [ ]:
def list_source_files(folder: Path) -> list[Path]:
    """List supported tabular files recursively in a source folder."""
    if not folder.is_dir():
        raise FileNotFoundError(f"No existe la carpeta de la fuente: {folder}")
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )


def load_tabular_file(path: Path) -> pd.DataFrame:
    """Load one CSV file and report errors with its path."""
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, encoding="utf-8-sig")
    raise ValueError(f"Formato no admitido para {path}; usa CSV.")


source_files = {
    source: list_source_files(folder)
    for source, folder in SOURCE_DIRS.items()
}
missing_sources = [source for source, files in source_files.items() if not files]
if missing_sources:
    expected = "\n".join(f"- {source}: {SOURCE_DIRS[source]}" for source in missing_sources)
    raise FileNotFoundError(
        "No se encontraron archivos CSV para todas las fuentes del proyecto. "
        "Agrega los CSV requeridos; fuentes sin archivos:\n" + expected
    )

dataframes = {}
for source, files in source_files.items():
    for path in files:
        relative_path = path.relative_to(SOURCE_DIRS[source]).with_suffix("")
        name = f"{source} / {relative_path.as_posix()}"
        dataframes[name] = load_tabular_file(path)

print(f"Archivos cargados correctamente: {len(dataframes)}")
for name, dataframe in dataframes.items():
    print(f"- {name}: {dataframe.shape[0]} filas × {dataframe.shape[1]} columnas")

## Inspección estructural

Cada archivo se conserva en un DataFrame independiente. Para limitar la exposición accidental de identificadores directos en las muestras y descripciones categóricas, se redactan columnas cuyos nombres sugieren Steam ID, Discord ID, license, citizen ID, token o claves de jugador/usuario. Esto no sustituye la anonimización del archivo de origen.

In [ ]:
SENSITIVE_NAME_PARTS = (
    "steam", "discord", "license", "citizenid", "citizen_id", "token",
    "player_id", "playerid", "user_id", "userid", "identifier",
)


def redacted_view(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return a display copy with likely direct identifiers masked."""
    result = dataframe.copy()
    for column in result.columns:
        normalized = re.sub(r"[^a-z0-9]+", "_", str(column).lower())
        if any(part in normalized for part in SENSITIVE_NAME_PARTS):
            result[column] = "[redactado]"
    return result


structure_rows = []
missing_rows = []
for name, dataframe in dataframes.items():
    print(f"\n## {name}")
    print("head() — primeros registros (vista redactada):")
    display(redacted_view(dataframe.head()))
    print("tail() — últimos registros (vista redactada):")
    display(redacted_view(dataframe.tail()))
    print("shape:", dataframe.shape)
    print("columns:", dataframe.columns.tolist())
    print("dtypes:")
    display(dataframe.dtypes.rename("dtype").to_frame())
    print("info():")
    dataframe.info()
    print("describe() — variables numéricas:")
    display(dataframe.describe())

    object_columns = [
        column for column in dataframe.columns
        if pd.api.types.is_object_dtype(dataframe[column].dtype)
        or pd.api.types.is_string_dtype(dataframe[column].dtype)
    ]
    if len(object_columns):
        text_data = redacted_view(dataframe[object_columns])
        if all(pd.api.types.is_object_dtype(dataframe[column].dtype) for column in object_columns):
            print("describe(include='object') — variables categóricas (vista redactada):")
            display(text_data.describe(include="object"))
        else:
            print("describe() — variables de texto (vista redactada):")
            display(text_data.describe())
    else:
        print("No hay columnas de tipo object para describe(include='object').")

    structure_rows.append({
        "Dataset": name,
        "Filas": dataframe.shape[0],
        "Columnas": dataframe.shape[1],
        "Tipos de datos": ", ".join(sorted(map(str, dataframe.dtypes.unique()))),
    })
    null_counts = dataframe.isna().sum()
    for column, count in null_counts[null_counts.gt(0)].items():
        missing_rows.append({
            "Dataset": name,
            "Columna": str(column),
            "Valores faltantes": int(count),
            "Porcentaje": round(float(count / len(dataframe) * 100), 2) if len(dataframe) else 0.0,
        })

structure = pd.DataFrame(structure_rows)
missing_summary = pd.DataFrame(
    missing_rows,
    columns=["Dataset", "Columna", "Valores faltantes", "Porcentaje"],
)
display(structure)
print("Valores faltantes por dataset y columna:")
display(missing_summary if not missing_summary.empty else pd.DataFrame({"Resultado": ["No se observaron valores faltantes en los archivos cargados."]}))

## Indicadores iniciales y lectura crítica

El conteo de duplicados siguiente identifica filas completamente iguales; no prueba que una fila repetida sea un error. Confirma claves y reglas de negocio antes de proponer su eliminación. La inspección tampoco demuestra por sí sola que las ubicaciones, duraciones, fechas o movimientos económicos sean semánticamente válidos.

In [ ]:
NUMERIC_RULES = {
    "duration_minutes": (1, None),
    "amount": (0, None),
    "online_players": (0, 64),
    "max_players": (1, None),
    "server_fps": (0, None),
    "cpu_percent": (0, 100),
    "memory_percent": (0, 100),
    "Age": (0, None),
    "PlayTimeHours": (0, None),
    "InGamePurchases": (0, 1),
    "SessionsPerWeek": (0, None),
    "AvgSessionDurationMinutes": (0, None),
    "PlayerLevel": (0, None),
    "AchievementsUnlocked": (0, None),
}
DATE_COLUMNS = {"connected_at", "disconnected_at", "transaction_at", "event_at", "snapshot_at"}
quality_rows = []
finding_rows = []

for name, dataframe in dataframes.items():
    duplicate_count = int(dataframe.duplicated().sum())
    quality_rows.append({
        "Dataset": name,
        "Filas duplicadas exactas": duplicate_count,
        "Columnas con faltantes": int(dataframe.isna().any().sum()),
        "Columnas numéricas": int(len(dataframe.select_dtypes(include="number").columns)),
        "Columnas texto": sum(
            pd.api.types.is_object_dtype(dataframe[column].dtype)
            or pd.api.types.is_string_dtype(dataframe[column].dtype)
            for column in dataframe.columns
        ),
    })
    if duplicate_count:
        finding_rows.append({
            "Dataset": name, "Variable": "Fila completa", "Problema": "Duplicado exacto",
            "Registros": duplicate_count, "Evidencia/regla": "dataframe.duplicated()",
        })

    for column in dataframe.columns:
        missing_count = int(dataframe[column].isna().sum())
        if missing_count:
            finding_rows.append({
                "Dataset": name, "Variable": str(column), "Problema": "Valor faltante",
                "Registros": missing_count, "Evidencia/regla": "isna()",
            })

        if column in NUMERIC_RULES:
            numeric = pd.to_numeric(dataframe[column], errors="coerce")
            non_numeric_count = int((dataframe[column].notna() & numeric.isna()).sum())
            if non_numeric_count:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Tipo/valor no numérico",
                    "Registros": non_numeric_count, "Evidencia/regla": "to_numeric(errors='coerce')",
                })
            minimum, maximum = NUMERIC_RULES[column]
            outside = pd.Series(False, index=dataframe.index)
            if minimum is not None:
                outside |= numeric.lt(minimum)
            if maximum is not None:
                outside |= numeric.gt(maximum)
            outside_count = int(outside.sum())
            if outside_count:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Fuera del rango orientador",
                    "Registros": outside_count,
                    "Evidencia/regla": f"mínimo={minimum}; máximo={maximum}",
                })

        if column in DATE_COLUMNS:
            parsed_dates = pd.to_datetime(dataframe[column], errors="coerce")
            invalid_dates = int((dataframe[column].notna() & parsed_dates.isna()).sum())
            if invalid_dates:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Fecha/hora no interpretable",
                    "Registros": invalid_dates, "Evidencia/regla": "to_datetime(errors='coerce')",
                })

    if {"connected_at", "disconnected_at"}.issubset(dataframe.columns):
        connected = pd.to_datetime(dataframe["connected_at"], errors="coerce")
        disconnected = pd.to_datetime(dataframe["disconnected_at"], errors="coerce")
        reversed_sessions = int((connected.notna() & disconnected.notna() & disconnected.lt(connected)).sum())
        if reversed_sessions:
            finding_rows.append({
                "Dataset": name, "Variable": "connected_at/disconnected_at",
                "Problema": "Desconexión anterior a conexión", "Registros": reversed_sessions,
                "Evidencia/regla": "disconnected_at < connected_at",
            })

quality_summary = pd.DataFrame(quality_rows)
findings = pd.DataFrame(
    finding_rows,
    columns=["Dataset", "Variable", "Problema", "Registros", "Evidencia/regla"],
)
display(quality_summary)
print("Hallazgos automáticos para revisión; los rangos son reglas orientadoras, no reglas universales del dominio:")
display(findings if not findings.empty else pd.DataFrame({"Resultado": ["No se detectaron condiciones mediante las reglas ejecutadas."]}))

print("\nInterpretación requerida antes de preparar los datos:")
print("1. Contrastar fechas, unidades y definiciones de variables con la documentación de cada fuente.")
print("2. Validar las reglas de rango orientadoras contra el esquema y el significado de cada fuente.")
print("3. Distinguir faltantes legítimos de errores de captura y justificar el tratamiento por variable.")
print("4. Confirmar periodo, población y licencia del dataset de Kaggle antes de compararlo con FiveM.")
print("5. Registrar en reports/informe_diagnostico.md la evidencia, impacto y acción propuesta por hallazgo.")

## Cierre

Este Notebook realiza un diagnóstico inicial, no una limpieza completa ni un análisis causal. Los conteos y resúmenes impresos deben trasladarse al informe junto con una interpretación basada en el dominio. Antes de guardar o publicar el Notebook, revisa y borra cualquier salida que pudiera exponer datos sensibles.